In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
import sklearn.metrics
import math
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

# Read Data

In [ ]:
train_data = pd.read_csv("/kaggle/input/dataset/covid-19-m-rna-vaccine-degradation-batch-6/train.csv")
test_data = pd.read_csv("/kaggle/input/dataset/covid-19-m-rna-vaccine-degradation-batch-6/test.csv")

# Preprocessing

In [ ]:
# Function to extract the numeric position from id_seqpos
def extract_position(id_seqpos):
    return int(id_seqpos.split('_')[-1])

# Add a new column for the numeric position
train_data['position'] = train_data['id_seqpos'].apply(extract_position)

# Now sort by 'id' and then by the numeric position
sorted_df = train_data.sort_values(['id', 'position'])

# Reset the index for easier viewing
sorted_df = sorted_df.reset_index(drop=True)

# Display the first few rows of the correctly sorted dataframe
print(sorted_df.head(20))

# Check the range of positions for a few ids
for id in sorted_df['id'].unique()[:5]:
    id_data = sorted_df[sorted_df['id'] == id]
    print(f"ID: {id}")
    print(f"Position range: {id_data['position'].min()} to {id_data['position'].max()}")
    print(f"Number of positions: {len(id_data)}")
    print(f"Missing positions: {set(range(id_data['position'].min(), id_data['position'].max()+1)) - set(id_data['position'])}\n")

In [ ]:
def get_id_seqpos(id_seq):
    z = id_seq[-2:]
    if z.isdigit():
        return int(z)
    else:
        return int(z[-1])
sorted_df["id_seqpos"] = sorted_df["id_seqpos"].apply(lambda x: get_id_seqpos(x))
#train_data["position"] = train_data["id_seqpos"].apply(lambda x: get_id_seqpos(x))

In [ ]:
%time
c=1
c2=0
for i in range(len(sorted_df["b5_sequence"])):
    if sorted_df.loc[i, "b5_sequence"] == "-1":
        #if (i != (len(sorted_df)-1)):
        if (i != 0):
            #e = len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])
            if (sorted_df.loc[i, "id"] != sorted_df.loc[i-1, "id"]) or \
            int((sorted_df.loc[i, "id_seqpos"]) == (len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1)):
                continue
            #elif int(sorted_df.loc[i, "id_seqpos"]) in range(int(sorted_df.loc[e+i, "id_seqpos"])+1, len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])):
            if int(sorted_df.loc[i-1, "id_seqpos"]) == c-1:
                c2+=1
                sorted_df.loc[i, "b5_sequence"] = sorted_df.loc[i-1, "sequence"]
                sorted_df.loc[i, "b5_structure"] = sorted_df.loc[i-1, "structure"]
                sorted_df.loc[i, "b5_predicted_loop_type"] = sorted_df.loc[i-1, "predicted_loop_type"]
            else:
                c+=1
                if c == len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1:
                    c = int(sorted_df.loc[len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]), "id_seqpos"])

    c+=1
    if c == len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1:
        c = int(sorted_df.loc[len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]), "id_seqpos"])

In [ ]:
sorted_df

In [ ]:
len(sorted_df[sorted_df["id"] == sorted_df.loc[0, "id"]])

In [ ]:
max_values_per_id = sorted_df.groupby("id")['id_seqpos'].max()
max_values_dict = max_values_per_id.to_dict()
max_values_dict

In [ ]:
sorted_df

In [ ]:
c=1
c2=0
j=0
print("c1 : ", c)
#print(c, sorted_df.loc[len(sorted_df[sorted_df["id"] == sorted_df.loc[c2, "id"]]), "id_seqpos"])
for i in range(1, 200)    #if sorted_df.loc[i, "b5_sequence"] == "-1":
    #if (i != 0):
    print("out : ", int(sorted_df.loc[i-1, "id_seqpos"]), c-1)
    if int(sorted_df.loc[i-1, "id_seqpos"]) == c-1:
        print("in1 : ", int(sorted_df.loc[i-1, "id_seqpos"]), c-1)
        #print("c1 : ", c)
        #print(sorted_df.loc[i, "b5_sequence"])
        print()
        
    else:
        if int(sorted_df.loc[i-1, "id_seqpos"]) == c:
            print("in2 : ", int(sorted_df.loc[i-1, "id_seqpos"]), c)
            print()
            
        if c == max_values_dict[sorted_df.loc[j, "id"]]:
            print("1 : ", c, max_values_dict[sorted_df.loc[j, "id"]])
            j+=1
            c2 += len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])
            c = int(sorted_df.loc[c2, "id_seqpos"])
            #print("c3 : ", c)
        c+=1

    #print(c)
    if c == max_values_dict[sorted_df.loc[j, "id"]]:
        print("2 : ", c, max_values_dict[sorted_df.loc[j, "id"]])
        j+=1
        #print("i : ", i)`
        c2 += len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])
        c = int(sorted_df.loc[c2, "id_seqpos"])
        #print("c2 : ", c)
    c+=1

In [ ]:
c=1
c2=0
for i in range(150):
    if sorted_df.loc[i, "b5_sequence"] == "-1":
        if (i != 0):
            #print("out : ", int(sorted_df.loc[i-1, "id_seqpos"]), c-1)
            if int(sorted_df.loc[i-1, "id_seqpos"]) == c-1:
                    #print("in1 : ", int(sorted_df.loc[i-1, "id_seqpos"]), c-1)
                    #print(sorted_df.loc[i, "b5_sequence"])
                    print()
            else:
                c+=1
                if int(sorted_df.loc[i-1, "id_seqpos"]) == c-1:
                    #print("in2 : ", int(sorted_df.loc[i-1, "id_seqpos"]), c-1)
                    print()
                if c == len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1:
                    #print(c, len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1)
                    c2 += len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])
                    c = int(sorted_df.loc[c2, "id_seqpos"])
                    #print("c2 : ", c)
    c+=1
    print("c1 : ", c)
    if c == len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1:
        #print(c, len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]]) - 1)
        c2 += len(sorted_df[sorted_df["id"] == sorted_df.loc[i, "id"]])
        c = int(sorted_df.loc[c2, "id_seqpos"])
        #print("c2 : ", c)

In [ ]:
c2

In [ ]:
l = train_data[train_data["b5_sequence"] == "-1"]
len(l)

In [ ]:
r = sorted_df[sorted_df["b5_sequence"] == "-1"]
len(r)

In [ ]:
# Function to display sequence and structure for an id, highlighting gaps
def display_seq_struct(id):
    id_data = sorted_df[sorted_df['id'] == id].sort_values('position')
    full_length = id_data['position'].max() + 1
    sequence = ['_'] * full_length
    structure = ['_'] * full_length
    for _, row in id_data.iterrows():
        sequence[row['position']] = row['sequence']
        structure[row['position']] = row['structure']
    print(f"ID: {id}")
    print("Sequence:  ", ''.join(sequence))
    print("Structure: ", ''.join(structure))
    print("Gaps at positions:", [i for i, s in enumerate(sequence) if s == '_'])
    print("\n")

# Display for the first few ids
for id in sorted_df['id'].unique()[:5]:
    display_seq_struct(id)

In [ ]:
sorted_df["sequence"]

In [ ]:
# label_encoding
sequence_encoding_map = {'A': 0, 'G' : 1, 'C' : 3, 'U' : 2}
structure_encoding_map = {'.' : 0, '(' : 1, ')' : 1}
looptype_encoding_map = {'S':6, 'E':2, 'H':0, 'I':5, 'X':4, 'M':3, 'B':1}

sequences_cols = ["sequence", 'b1_sequence', 'a1_sequence', 'b2_sequence', 'a2_sequence', 'b3_sequence', 'a3_sequence', 'b4_sequence', 'a4_sequence', 'b5_sequence', 'a5_sequence']
structure_cols = ["structure", 'b1_structure', 'a1_structure', 'b2_structure', 'a2_structure', 'b3_structure', 'a3_structure', 'b4_structure', 'a4_structure', 'b5_structure', 'a5_structure']
predicted_loop_type_cols = ["predicted_loop_type", 'b1_predicted_loop_type', 'a1_predicted_loop_type', 'b2_predicted_loop_type', 'a2_predicted_loop_type', 'b3_predicted_loop_type', 'a3_predicted_loop_type', 'b4_predicted_loop_type', 'a4_predicted_loop_type', 'b5_predicted_loop_type', 'a5_predicted_loop_type']

'''# Function to encode columns
def encode_column(column, encoding_map):
    return column.apply(lambda x: [encoding_map.get(char, -1) for char in x])'''

# Apply encoding
for col in sequences_cols:
    train_data[col] = train_data[col].apply(lambda x: sequence_encoding_map[x] if x in sequence_encoding_map else -1)
    
for col in structure_cols:
    train_data[col] = train_data[col].apply(lambda x: structure_encoding_map[x] if x in structure_encoding_map else -1)

for col in predicted_loop_type_cols:
    train_data[col] = train_data[col].apply(lambda x: looptype_encoding_map[x] if x in looptype_encoding_map else -1)

train_data.head()

# Modeling

In [ ]:
'''c=0
threshold = 0.01
for col in train_data.columns:
    if train_data[col].dtype in ['float64', 'int64']:
        #boxplot beofore removing the ooutliers
        sns.boxplot(x=col, data=train_data)
        plt.show()
        # Calculate quartiles and IQR
        print(col)
        Q1 =train_data[col].quantile(0.25)
        Q3 = train_data[col].quantile(0.75)
        IQR = Q3 - Q1

        # Identify outliers
        outliers = (train_data[col] < (Q1 - 1.5 * IQR)) | (train_data[col] > (Q3 + 1.5 * IQR))

        # Remove outliers
        covid_no_outliers = train_data[~outliers]

        # Plot boxplot without outliers
        sns.boxplot(x=col, data=covid_no_outliers)
        plt.show()
        '''

In [ ]:
X = train_data.drop(["id","id_seqpos", "reactivity", "deg_Mg_pH10", "deg_Mg_50C"], axis=1)
y = train_data[["reactivity", "deg_Mg_pH10", "deg_Mg_50C"]]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 79)

def rmse(y_actual, y_pred):
    mse = sklearn.metrics.mean_squared_error(y_actual, y_pred)  
    rmse = math.sqrt(mse)  
    return rmse

def mcrmse(y_actual, y_pred):
    score=0
    num_scored=3
    for i in range(num_scored):
        score += rmse(y_actual.iloc[:, i], y_pred[:,  i]) / num_scored
    return score

xgb = MultiOutputRegressor(XGBRegressor(learning_rate=0.2,n_estimators=1500,max_depth=4,min_child_weight = 1,gamma=0,nthread =4,scale_pos_weight =1,seed =35))
xgb.fit(X_train, y_train)
yhat = xgb.predict(X_val)
yt_hat = xgb.predict(X_train)
c1=mcrmse(y_train, yt_hat)
c=mcrmse(y_val, yhat)
print(c1)
print("-----------")
print(c)

In [ ]:
'''xgb = MultiOutputRegressor(XGBRegressor(learning_rate=0.2,n_estimators=1500,max_depth=4,min_child_weight = 1,gamma=0,nthread =4,scale_pos_weight =1,seed =35))
xgb.fit(X_train, y_train)
yhat = xgb.predict(X_val)
yt_hat = xgb.predict(X_train)
c1=mcrmse(y_train, yt_hat)
c=mcrmse(y_val, yhat)
print(c1)
print("-----------")
print(c)'''

# another solution

In [ ]:
'''train_data = train_data.sort_values(by=['id_seqpos'], key=lambda x: x.str.extract(r'_(\d+)$').astype(int).iloc[:, 0])

a = ['b1_structure','b2_structure','b3_structure','b4_structure','b5_structure']
for i in a:
   train_data[i] = train_data[i].replace({'.':2 , '(': 0 , ')':1,'-1':-1})
b = ['b1_sequence','b2_sequence','b3_sequence','b4_sequence','b5_sequence']
for e in b:
   train_data[e] = train_data[e].replace({'A':3 , 'C': 4 , 'G':5,'U':6,'-1':-1})
c = ['b1_predicted_loop_type','b2_predicted_loop_type','b3_predicted_loop_type','b4_predicted_loop_type','b5_predicted_loop_type']
for t in c:
   train_data[t] = train_data[t].replace({'B':7 , 'E': 8 , 'H':9,'I':10,'M':11,'S':12,'X':13,'-1':-1})
d = ['sequence','a1_sequence','a2_sequence','a3_sequence','a4_sequence','a5_sequence']
for q in d:   
  train_data[q] = train_data[q].replace({'A':3 , 'C': 4 , 'G':5,'U':6})
e = ['structure','a1_structure','a2_structure','a3_structure','a4_structure','a5_structure']  
for p in e:
  train_data[p] = train_data[p].replace({'.':2 , '(': 0 , ')':1})
f = ['predicted_loop_type','a1_predicted_loop_type','a1_predicted_loop_type','a2_predicted_loop_type','a3_predicted_loop_type','a4_predicted_loop_type','a5_predicted_loop_type']  
for o in f:
  train_data[o] = train_data[o].replace({'B':7 , 'E': 8 , 'H':9,'I':10,'M':11,'S':12,'X':13})

X = train_data.drop(['id','id_seqpos','reactivity','deg_Mg_pH10','deg_Mg_50C'],axis=1)
y = train_data[['reactivity', 'deg_Mg_pH10', 'deg_Mg_50C']]'''

In [ ]:
#train_data = train_data.sort_values(by=['id_seqpos'], key=lambda x: x.str.extract(r'_(\d+)$').astype(int).iloc[:, 0])


a = ['b1_structure','b2_structure','b3_structure','b4_structure','b5_structure']
for i in a:
   train_data[i] = train_data[i].replace({'.':0 , '(': 1 , ')':1,'-1':-1})
b = ['b1_sequence','b2_sequence','b3_sequence','b4_sequence','b5_sequence']
for e in b:
   train_data[e] = train_data[e].replace({'A':0 , 'C': 1 , 'G':2,'U':3,'-1':-1})
c = ['b1_predicted_loop_type','b2_predicted_loop_type','b3_predicted_loop_type','b4_predicted_loop_type','b5_predicted_loop_type']
for t in c:
   train_data[t] = train_data[t].replace({'B':0 , 'E': 1 , 'H':2,'I':3,'M':4,'S':5,'X':6,'-1':-1})
d = ['sequence','a1_sequence','a2_sequence','a3_sequence','a4_sequence','a5_sequence']
for q in d:   
  train_data[q] = train_data[q].replace({'A':0 , 'C': 1 , 'G':2,'U':3})
e = ['structure','a1_structure','a2_structure','a3_structure','a4_structure','a5_structure']  
for p in e:
  train_data[p] = train_data[p].replace({'.':0 , '(': 1 , ')':1})
f = ['predicted_loop_type','a1_predicted_loop_type','a1_predicted_loop_type','a2_predicted_loop_type','a3_predicted_loop_type','a4_predicted_loop_type','a5_predicted_loop_type']  
for o in f:
  train_data[o] = train_data[o].replace({'B':0 , 'E': 1 , 'H':2,'I':3,'M':4,'S':5,'X':6})

X = train_data.drop(['id','id_seqpos','reactivity','deg_Mg_pH10','deg_Mg_50C'],axis=1)
y = train_data[['reactivity', 'deg_Mg_pH10', 'deg_Mg_50C']]

In [ ]:
from sklearn.model_selection import train_test_split , KFold
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.18, random_state=40)

In [ ]:
from xgboost import XGBRegressor
xgb_model = MultiOutputRegressor(XGBRegressor(max_depth=6,learning_rate=0.18,colsample_bytree=0.85,reg_alpha=5,reg_lambda=6
                      ,min_child_weight = 2,n_estimators=1500))


xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

def MCRMSE_numpy(y_true, y_pred):
    colwise_mse = np.mean(np.square(y_true - y_pred), axis=1)
    return np.mean(np.sqrt(colwise_mse))

print(MCRMSE_numpy(y_test,y_pred_xgb))
print(xgb_model.score(X_train,y_train) * 100)
print(xgb_model.score(X_test,y_test) * 100)
     

In [ ]:
from sklearn.metrics import make_scorer
def mcrmse (y_true,y_pred,N=3):
  assert len(y_true) == len(y_pred)
  n = len(y_true)
  return np.sum(np.sqrt(np.sum((y_true - y_pred)**2,axis=0)/n)) / N
aa = make_scorer(mcrmse , greater_is_better = False)
print(mcrmse(xgb_model.predict(X_train) , np.array(y_train)))
print(mcrmse(xgb_model.predict(X_test) , np.array(y_test)))

In [ ]:
'''def rmse(y_actual, y_pred):
    mse = sklearn.metrics.mean_squared_error(y_actual, y_pred)  
    rmse = math.sqrt(mse)  
    return rmse

def mcrmse1(y_actual, y_pred):
    score=0
    num_scored=3
    for i in range(num_scored):
        score += rmse(y_actual.iloc[:, i], y_pred[:,  i]) / num_scored
    return score'''

In [ ]:
'''#c=mcrmse(y_test, yhat)
print(mcrmse1((y_train), xgb_model.predict(X_train)))
print(mcrmse1((y_test), xgb_model.predict(X_test)))'''

In [ ]:
a = ['b1_structure','b2_structure','b3_structure','b4_structure','b5_structure']
for i in a:
   test_data[i] = test_data[i].replace({'.':2 , '(': 0 , ')':1,'-1':-1})
b = ['b1_sequence','b2_sequence','b3_sequence','b4_sequence','b5_sequence']
for e in b:
   test_data[e] = test_data[e].replace({'A':3 , 'C': 4 , 'G':5,'U':6,'-1':-1})
c = ['b1_predicted_loop_type','b2_predicted_loop_type','b3_predicted_loop_type','b4_predicted_loop_type','b5_predicted_loop_type']
for t in c:
   test_data[t] = test_data[t].replace({'B':7 , 'E': 8 , 'H':9,'I':10,'M':11,'S':12,'X':13,'-1':-1})
d = ['sequence','a1_sequence','a2_sequence','a3_sequence','a4_sequence','a5_sequence']
for q in d:   
  test_data[q] = test_data[q].replace({'A':3 , 'C': 4 , 'G':5,'U':6})
e = ['structure','a1_structure','a2_structure','a3_structure','a4_structure','a5_structure']  
for p in e:
  test_data[p] = test_data[p].replace({'.':2 , '(': 0 , ')':1})
f = ['predicted_loop_type','a1_predicted_loop_type','a1_predicted_loop_type','a2_predicted_loop_type','a3_predicted_loop_type','a4_predicted_loop_type','a5_predicted_loop_type']  
for o in f:
  test_data[o] = test_data[o].replace({'B':7 , 'E': 8 , 'H':9,'I':10,'M':11,'S':12,'X':13})

In [ ]:
'''a = ['b1_structure','b2_structure','b3_structure','b4_structure','b5_structure']
for i in a:
   test_data[i] = test_data[i].replace({'.':0 , '(': 1 , ')':1,'-1':-1})
b = ['b1_sequence','b2_sequence','b3_sequence','b4_sequence','b5_sequence']
for e in b:
   test_data[e] = test_data[e].replace({'A':0 , 'C':1 , 'G':2,'U':3,'-1':-1})
c = ['b1_predicted_loop_type','b2_predicted_loop_type','b3_predicted_loop_type','b4_predicted_loop_type','b5_predicted_loop_type']
for t in c:
   test_data[t] = test_data[t].replace({'B':0 , 'E':1 , 'H':2,'I':3,'M':4,'S':5,'X':6,'-1':-1})
d = ['sequence','a1_sequence','a2_sequence','a3_sequence','a4_sequence','a5_sequence']
for q in d:   
  test_data[q] = test_data[q].replace({'A':0 , 'C': 1 , 'G':2,'U':3})
e = ['structure','a1_structure','a2_structure','a3_structure','a4_structure','a5_structure']  
for p in e:
  test_data[p] = test_data[p].replace({'.':0 , '(': 1 , ')':1})
f = ['predicted_loop_type','a1_predicted_loop_type','a1_predicted_loop_type','a2_predicted_loop_type','a3_predicted_loop_type','a4_predicted_loop_type','a5_predicted_loop_type']  
for o in f:
  test_data[o] = test_data[o].replace({'B':0 , 'E': 1 , 'H':2,'I':3,'M':4,'S':5,'X':6})'''

# TEST

In [ ]:
ID = test_data['id_seqpos']
TEST = test_data.drop(['id_seqpos','id'],axis=1)
a = xgb_model.predict(TEST)
new = pd.DataFrame(a,columns=['reactivity','deg_Mg_pH10','deg_Mg_50C'])

new_data = pd.concat([ID.rename('id_seqpos'),new], axis=1)

new_data.to_csv('xgboost_new_tunning.csv',index=False)